In [1]:
!pip install -q transformers

In [2]:
import torch
from transformers import ViTImageProcessor, ViTForImageClassification

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.10.0+cu128
CUDA available: True


In [3]:
# Load ViT image processor
processor = ViTImageProcessor.from_pretrained(
    "google/vit-base-patch16-224"
)

# Load pretrained ViT model
vit_model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224"
)

print("ViT model loaded successfully!")

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViT model loaded successfully!


In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

vit_model = vit_model.to(device)

print("Using device:", device)

Using device: cuda


In [5]:
import os

# Main ImageNet Mini dataset folder
dataset_folder = "/kaggle/input/datasets/ifigotin/imagenetmini-1000"

# Create an empty list to store image paths
image_files = []

# Search through all subfolders
for root, dirs, files in os.walk(dataset_folder):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
            image_files.append(
                os.path.join(root, file)
            )

print("Total images found:", len(image_files))

Total images found: 38668


In [6]:
# Show the full path of the first image
print(image_files[0])

/kaggle/input/datasets/ifigotin/imagenetmini-1000/imagenet-mini/val/n01531178/ILSVRC2012_val_00029581.JPEG


In [7]:
from tensorflow.keras.preprocessing import image

# Select the first image
image_path = image_files[0]

# Load the image
img = image.load_img(
    image_path,
    target_size=(224, 224)
)

print("Testing image:", os.path.basename(image_path))

Testing image: ILSVRC2012_val_00029581.JPEG


In [8]:
print("Testing image:", os.path.basename(image_path))

Testing image: ILSVRC2012_val_00029581.JPEG


In [9]:
# Process the image using the ViT processor
inputs = processor(
    images=img,
    return_tensors="pt"
)

print("Image processed successfully for ViT!")

Image processed successfully for ViT!


In [10]:
# Move the input tensors to GPU
inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

print("Input moved to:", device)

Input moved to: cuda


In [11]:
import torch

# Run the model without calculating gradients
with torch.no_grad():
    outputs = vit_model(**inputs)

# Get raw prediction scores
logits = outputs.logits

# Convert raw scores into probabilities
probabilities = torch.nn.functional.softmax(
    logits,
    dim=-1
)

# Get the top 3 predictions
top3 = torch.topk(
    probabilities,
    k=3
)

print("\nTop 3 ViT Predictions:")

for i in range(3):

    # Get ImageNet class ID
    class_id = top3.indices[0][i].item()

    # Get confidence score
    confidence = top3.values[0][i].item()

    # Get class name
    label = vit_model.config.id2label[class_id]

    print(
        f"{i+1}. {label} - "
        f"{confidence * 100:.2f}%"
    )


Top 3 ViT Predictions:
1. goldfinch, Carduelis carduelis - 99.77%
2. brambling, Fringilla montifringilla - 0.07%
3. house finch, linnet, Carpodacus mexicanus - 0.04%


In [12]:
import time
import torch

# Make sure GPU operations are synchronized before timing
if torch.cuda.is_available():
    torch.cuda.synchronize()

# Start timer
start_time = time.time()

# Run ViT prediction
with torch.no_grad():
    outputs = vit_model(**inputs)

# Wait for GPU to finish
if torch.cuda.is_available():
    torch.cuda.synchronize()

# End timer
end_time = time.time()

# Calculate inference time
inference_time = end_time - start_time

print("Image:", os.path.basename(image_path))
print("ViT Inference Time:", inference_time, "seconds")

Image: ILSVRC2012_val_00029581.JPEG
ViT Inference Time: 0.02345132827758789 seconds
